In [6]:
# %% 1. Setup
import os, sys, json
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = os.environ.get("RAG_ROOT", "/content/drive/MyDrive/rag_trust")   # holds data/scores.parquet, data/frozen/
REPO = "/content" # folder that holds eval_core.py
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
sys.path.insert(0, REPO)
import eval_core as C

D = pd.read_parquet(f"{ROOT}/data/scores.parquet")
TRF = f"{ROOT}/data/trustrag_flags.parquet"      # context_id, rank, trustrag_removed (from the TrustRAG cell)
if os.path.exists(TRF):
    D = D.merge(pd.read_parquet(TRF), on=["context_id", "rank"], how="left", validate="one_to_one")
T = json.load(open(f"{ROOT}/data/frozen/thresholds.json"))["thresholds"]
V = D[D["split"] == "val"]
print(len(V), "val rows | datasets:", sorted(V["dataset"].unique()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1390 val rows | datasets: ['hotpotqa', 'nq']


In [7]:
# %% 2. eval_core.evaluate vs sklearn / plain NumPy, on val rows relabelled as test (in memory only)
rows = [dict(r, split="test") for r in V.to_dict("records")]
g, n_, lab, gvar = V["group_variant"], V["n_poison"], V["label"], None
orig = V[V["group_variant"] == "original"]
sets = {
    "original":      (orig[(orig["n_poison"] == 1) & (orig["label"] == 1)], orig[(orig["n_poison"] == 0) & (orig["label"] == 0)]),
    "original_n5":   (orig[(orig["n_poison"] == 5) & (orig["label"] == 1)], orig[(orig["n_poison"] == 0) & (orig["label"] == 0)]),
    "collateral_n1": (orig.iloc[0:0],                                       orig[(orig["n_poison"] == 1) & (orig["label"] == 0)]),
}
ok = True
for m, col in (("featscorer", "s_featscorer"), ("length", "s_length"), ("retrieval", "s_retrieval")):
    thr = {t: T[m][t]["threshold"] for t in ("fpr5", "fpr1")}
    res = C.evaluate(rows, col, thr, n_boot=200)["in_dist"]          # CIs are not compared, so 200 resamples is enough
    for name, (P, N) in sets.items():
        r = res[name]
        assert r["n_pos"] == len(P) and r["n_neg"] == len(N), (m, name, r["n_pos"], r["n_neg"], len(P), len(N))
        mine = {}
        if len(P):
            mine["auroc"] = roc_auc_score(np.r_[np.ones(len(P)), np.zeros(len(N))], np.r_[P[col], N[col]])
        for t in ("fpr5", "fpr1"):
            if len(P):
                mine[f"tpr_{t}"] = float((P[col] > thr[t]).mean())
            mine[f"fpr_{t}"] = float((N[col] > thr[t]).mean())
        for k, v in mine.items():
            theirs = r[k]["value"]
            good = theirs is not None and abs(theirs - v) <= 5e-4          # 3 decimal places
            ok &= good
            if not good or (m == "featscorer" and name == "original"):
                print(f"{m:10} {name:13} {k:9} eval_core {theirs} | mine {v:.4f} {'ok' if good else '<-- DIFFERENT'}")
print("\neval_core matches sklearn and plain NumPy on val:", ok)

featscorer original      auroc     eval_core 0.9946 | mine 0.9946 ok
featscorer original      tpr_fpr5  eval_core 0.9798 | mine 0.9798 ok
featscorer original      fpr_fpr5  eval_core 0.05 | mine 0.0500 ok
featscorer original      tpr_fpr1  eval_core 0.8687 | mine 0.8687 ok
featscorer original      fpr_fpr1  eval_core 0.01 | mine 0.0100 ok

eval_core matches sklearn and plain NumPy on val: True


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# %% 3. pairing() against protocol section 7, on a synthetic row set (no real data involved)
def R(i, gv, n, v, lab, q="q1"):
    return dict(id=i, dataset="nq", qid=q, split="test", group_variant=gv, n_poison=n, variant=v, label=lab)

rows_t = [
    R("op1", "original", 1, "original", 1), R("op5", "original", 5, "original", 1),
    R("oc0", "original", 0, "original", 0), R("oc0_q2", "original", 0, "original", 0, q="q2"),
    R("oc1", "original", 1, "original", 0),
    R("pr1", "prefix_removed", 1, "prefix_removed", 1), R("pr5", "prefix_removed", 5, "prefix_removed", 1),
    R("pr_clean", "prefix_removed", 1, "original", 0),
    R("cp", "clean_prefixed", 0, "clean_prefixed", 0), R("cp_other", "clean_prefixed", 0, "original", 0),
    R("rw", "rewritten_clean", 0, "rewritten_clean", 0), R("rw_other", "rewritten_clean", 0, "original", 0),
    R("sp", "shortcut_controlled", 1, "prefix_removed", 1), R("sc0", "shortcut_controlled", 0, "clean_truncated", 0),
    R("sc1", "shortcut_controlled", 1, "clean_truncated", 0), R("sc_unmatched", "shortcut_controlled", 0, "original", 0),
    R("inj", "injection", 0, "injection", 1), R("inj_other", "injection", 0, "original", 0),
]
expected = {   # name: (positive ids, negative ids)
    "original":            ({"op1"}, {"oc0", "oc0_q2"}),
    "original_n5":         ({"op5"}, {"oc0", "oc0_q2"}),
    "prefix_poison_side_n5": ({"pr5"}, {"oc0", "oc0_q2"}),
    "prefix_poison_side":  ({"pr1"}, {"oc0", "oc0_q2"}),
    "prefix_clean_side":   ({"op1"}, {"cp"}),
    "llm_style":           ({"op1"}, {"rw"}),
    "shortcut_controlled": ({"sp"},  {"sc0"}),
    "unseen_attack":       ({"inj"}, {"oc0"}),              # negatives: n=0 clean of the same questions only (q1, not q2)
    "collateral_n1":       (set(),   {"oc1"}),
}
for name, (p, n) in expected.items():
    gp, gn = C.pairing(rows_t, name)
    assert {r["id"] for r in gp} == p and {r["id"] for r in gn} == n, f"{name}: {[r['id'] for r in gp]} / {[r['id'] for r in gn]}"
assert set(expected) == set(C.PAIRINGS), (f"in eval_core but not in expected: {set(C.PAIRINGS) - set(expected)} | "
                                         f"in expected but not in eval_core: {set(expected) - set(C.PAIRINGS)}")
print("pairing() matches protocol section 7 for all", len(expected), "pairings")
print("eval_core pairings:", C.PAIRINGS)
print("Protocol says n=1 and n=5 are reported separately: confirm with Samana that n=5 for prefix_clean_side "
      "and llm_style is left out on purpose.")

pairing() matches protocol section 7 for all 9 pairings
eval_core pairings: ['original', 'original_n5', 'prefix_poison_side', 'prefix_poison_side_n5', 'prefix_clean_side', 'llm_style', 'shortcut_controlled', 'unseen_attack', 'collateral_n1']
Protocol says n=1 and n=5 are reported separately: confirm with Samana that n=5 for prefix_clean_side and llm_style is left out on purpose.


In [10]:
# %% 4. Threshold rules: eval_core.threshold_at_fpr vs the frozen np.quantile thresholds (val clean n=0)
rv = V.to_dict("records")
negs = np.array(C.val_clean_n0(rv, "s_featscorer"))
for tag, fpr in (("fpr5", 0.05), ("fpr1", 0.01)):
    t_h, t_m = C.threshold_at_fpr(negs, fpr), T["featscorer"][tag]["threshold"]
    print(f"{tag}: val clean flagged out of {len(negs)}: eval_core rule {int((negs > t_h).sum())} | frozen threshold {int((negs > t_m).sum())}")

fpr5: val clean flagged out of 200: eval_core rule 10 | frozen threshold 10
fpr1: val clean flagged out of 200: eval_core rule 2 | frozen threshold 2


In [11]:
# %% 5. scores_features.jsonl: every val/test row, read by notebooks 05 and 06
P = pd.read_json(f"{ROOT}/data/passages.jsonl", lines=True)
P = P[P["split"].isin(["val", "test"]) & P["context_id"].notna()][["context_id", "rank", "passage_id", "group_variant"]]
TR = "trustrag_removed"                       # <- name of your per-passage TrustRAG removal column
cols = ["context_id", "rank", "s_featscorer", "s_retrieval", TR] + (["s_length"] if "s_length" in D else [])
assert set(cols) <= set(D.columns), f"missing columns: {set(cols) - set(D.columns)}"
Dk = D[D["context_id"].notna()][cols]
X = P.merge(Dk, on=["context_id", "rank"], how="left", validate="one_to_one")
miss = X[["s_featscorer", "s_retrieval", TR]].isna().any(axis=1)
print(len(P), "val/test rows |", int(miss.sum()), "without a score")
if miss.any(): print(X[miss].groupby("group_variant").size())
assert not miss.any(), "every val/test row (all group types) needs a score before writing"
os.makedirs(f"{ROOT}/step3_features", exist_ok=True)
out = f"{ROOT}/step3_features/scores_features.jsonl"
with open(out, "w") as f:
    for r in X.sort_values(["context_id", "rank"]).to_dict("records"):
        d = {"context_id": r["context_id"], "passage_id": r["passage_id"], "rank": int(r["rank"]),
             "featscorer": float(r["s_featscorer"]), "retrieval_score": float(r["s_retrieval"]),
             "trustrag_removed": int(bool(r[TR]))}
        if "s_length" in r: d["len_score"] = float(r["s_length"])
        f.write(json.dumps(d) + "\n")
print("wrote", out, "|", len(X), "rows")

20860 val/test rows | 0 without a score
wrote /content/drive/MyDrive/rag_trust/step3_features/scores_features.jsonl | 20860 rows
